# Daily Challenge – Week 7, Day 2: Multi-Attention & Transformer Comparisons
Run in **Google Colab with a GPU**. The notebook downloads the NLI dataset (*Contradictory, My Dear Watson*: premise + hypothesis → entailment / neutral / contradiction) from the course GitHub.

**Single-head vs. multi-head vs. cross-attention:**
- **Single-head self-attention:** each token computes *one* set of attention weights over all tokens in the *same* sequence.
- **Multi-head self-attention:** several heads run in parallel on smaller slices of the embedding. Each can focus on a different relationship (syntax, meaning, position), and their outputs are concatenated.
- **Cross-attention:** the queries come from **one** sequence and the keys/values from **another**, e.g. a translation decoder attending to the encoder's English sentence, or a hypothesis attending to a premise.

## Setup

In [ ]:
!pip install --quiet transformers datasets einops

In [ ]:
import os, glob, zipfile, urllib.request, math, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

URL = ("https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%206/"
       "W6D1%20GenAi%20France/Basics%20of%20BERT%20and%20XLM-RoBERTa%20-%20PyTorch%20-%202.zip")
if not glob.glob("nli_data/**/*.csv", recursive=True):
    urllib.request.urlretrieve(URL, "nli.zip")
    with zipfile.ZipFile("nli.zip") as z:
        z.extractall("nli_data")
csv_files = glob.glob("nli_data/**/*.csv", recursive=True)
print(csv_files)

train_file = next(f for f in csv_files if "train" in os.path.basename(f).lower())
df = pd.read_csv(train_file)
print(df.shape)
df.head()

In [ ]:
# Keep the English pairs to make the comparison simple (the dataset is multilingual)
if "lang_abv" in df.columns:
    df = df[df["lang_abv"] == "en"].reset_index(drop=True)
print("English pairs:", len(df))
print(df["label"].value_counts())   # 0 = entailment, 1 = neutral, 2 = contradiction
LABELS = ["entailment", "neutral", "contradiction"]

## 1. Single-Head Attention Implementation

In [ ]:
class Attention(nn.Module):
    """Single-head scaled dot-product attention with learned Q/K/V projections."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.q = nn.Linear(hidden_dim, hidden_dim)
        self.k = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, hidden_dim)
        self.scale = hidden_dim ** -0.5
        self.last_weights = None                     # kept for inspection

    def forward(self, x, mask=None):
        Q, K, V = self.q(x), self.k(x), self.v(x)                 # (batch, seq, hidden)
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale  # (batch, seq, seq)
        if mask is not None:                                       # mask: (batch, seq), 1 = real token
            scores = scores.masked_fill(mask[:, None, :] == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        self.last_weights = weights.detach()
        return torch.matmul(weights, V), weights

# Validate shapes with dummy tensors
batch, seq_len, hidden_dim = 2, 5, 16
dummy = torch.randn(batch, seq_len, hidden_dim)
single = Attention(hidden_dim)
out, w = single(dummy)
print("Input:", dummy.shape, "| Output:", out.shape, "| Attention weights:", w.shape)
print("Each row of weights sums to 1:", torch.allclose(w.sum(-1), torch.ones(batch, seq_len)))
print("Attention weights for the first example:\n", w[0].round(decimals=2))

## 2. Multi-Head Attention Module
Split the embedding into `num_heads` slices, run attention on each slice, concatenate, project back. Includes dropout and a residual connection.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, hidden_dim, num_heads, dropout=0.1, residual=True):
        super().__init__()
        self.residual = residual
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim)    # Q, K and V in one projection
        self.out = nn.Linear(hidden_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.last_weights = None

    def forward(self, x, mask=None):
        b, n, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)                       # each (b, n, d)
        # (b, n, d) → (b, heads, n, head_dim)
        q, k, v = [t.reshape(b, n, self.num_heads, self.head_dim).transpose(1, 2) for t in (q, k, v)]

        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)   # (b, heads, n, n)
        if mask is not None:
            scores = scores.masked_fill(mask[:, None, None, :] == 0, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        self.last_weights = weights.detach()

        context = self.dropout(weights) @ v                           # (b, heads, n, head_dim)
        context = context.transpose(1, 2).reshape(b, n, d)            # concatenate the heads
        out = self.dropout(self.out(context))
        return x + out if self.residual else out                       # residual connection

mha = MultiHeadAttention(hidden_dim=16, num_heads=4)
y = mha(dummy)
print("Input:", dummy.shape, "→ output:", y.shape)
print("Per-head attention weights:", mha.last_weights.shape, "(batch, heads, seq, seq)")

## 3. (Optional) Custom Encoder Stack & Training Loop

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")   # we reuse this tokenizer only
MAX_LEN = 96

def encode(frame):
    enc = tokenizer(list(frame["premise"]), list(frame["hypothesis"]),
                    padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"], torch.tensor(frame["label"].values)

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
train_data = TensorDataset(*encode(train_df))
val_data = TensorDataset(*encode(val_df))
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64)
print("Train:", len(train_data), "| Val:", len(val_data))

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, hidden_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.attn = MultiHeadAttention(hidden_dim, num_heads, dropout, residual=False)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.ff = nn.Sequential(nn.Linear(hidden_dim, ff_dim), nn.GELU(), nn.Dropout(dropout),
                                nn.Linear(ff_dim, hidden_dim))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = x + self.attn(self.norm1(x), mask)             # pre-norm attention + residual
        return x + self.dropout(self.ff(self.norm2(x)))    # feed-forward + residual


class CustomEncoderClassifier(nn.Module):
    def __init__(self, vocab_size, hidden_dim=128, num_heads=4, num_layers=2, num_classes=3, max_len=MAX_LEN):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, hidden_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, hidden_dim)           # learned positional encoding
        self.blocks = nn.ModuleList([EncoderBlock(hidden_dim, num_heads, hidden_dim * 4) for _ in range(num_layers)])
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Linear(hidden_dim, num_classes)

    def forward(self, ids, mask):
        pos = torch.arange(ids.size(1), device=ids.device)
        x = self.tok_emb(ids) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x, mask)
        x = self.norm(x)
        pooled = (x * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True)   # mean over real tokens
        return self.head(pooled)

custom = CustomEncoderClassifier(tokenizer.vocab_size).to(device)
print(f"Custom encoder parameters: {sum(p.numel() for p in custom.parameters()):,}")

In [ ]:
def evaluate_custom(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for ids, mask, labels in loader:
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            correct += (model(ids, mask).argmax(-1) == labels).sum().item()
            total += len(labels)
    return correct / total

optimizer = torch.optim.AdamW(custom.parameters(), lr=3e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()
custom_log = []
start = time.time()
for epoch in range(8):
    custom.train()
    total_loss = 0
    for ids, mask, labels in train_loader:
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(custom(ids, mask), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
    val_acc = evaluate_custom(custom, val_loader)
    custom_log.append((total_loss / len(train_data), val_acc))
    print(f"Epoch {epoch + 1}: train loss {custom_log[-1][0]:.4f} | val accuracy {val_acc:.3f}")
custom_time = time.time() - start

## 4. Fine-tuned Transformer Baseline (DistilBERT)

In [ ]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup

bert = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3).to(device)
print(f"DistilBERT parameters: {sum(p.numel() for p in bert.parameters()):,}")

opt = torch.optim.AdamW(bert.parameters(), lr=3e-5, weight_decay=0.01)
EPOCHS = 3
scheduler = get_linear_schedule_with_warmup(opt, int(0.1 * EPOCHS * len(train_loader)), EPOCHS * len(train_loader))

def evaluate_bert(loader):
    bert.eval()
    correct = total = 0
    with torch.no_grad():
        for ids, mask, labels in loader:
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            correct += (bert(input_ids=ids, attention_mask=mask).logits.argmax(-1) == labels).sum().item()
            total += len(labels)
    return correct / total

bert_log = []
start = time.time()
for epoch in range(EPOCHS):
    bert.train()
    total_loss = 0
    for ids, mask, labels in train_loader:
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
        opt.zero_grad()
        out = bert(input_ids=ids, attention_mask=mask, labels=labels)
        out.loss.backward()
        opt.step()
        scheduler.step()
        total_loss += out.loss.item() * len(labels)
    val_acc = evaluate_bert(val_loader)
    bert_log.append((total_loss / len(train_data), val_acc))
    print(f"Epoch {epoch + 1}: train loss {bert_log[-1][0]:.4f} | val accuracy {val_acc:.3f}")
bert_time = time.time() - start

In [ ]:
results = pd.DataFrame({
    "model": ["Custom attention encoder", "DistilBERT (fine-tuned)"],
    "parameters": [sum(p.numel() for p in custom.parameters()), sum(p.numel() for p in bert.parameters())],
    "best val accuracy": [max(a for _, a in custom_log), max(a for _, a in bert_log)],
    "training time (s)": [round(custom_time), round(bert_time)],
})
print("Majority-class baseline accuracy:", round(val_df["label"].value_counts(normalize=True).max(), 3))
results

## 5. Attention Maps

In [ ]:
def show_custom_attention(idx=0, layer=0):
    ids, mask, label = val_data[idx]
    custom.eval()
    with torch.no_grad():
        custom(ids[None].to(device), mask[None].to(device))
    n = int(mask.sum())
    tokens = tokenizer.convert_ids_to_tokens(ids[:n])
    weights = custom.blocks[layer].attn.last_weights[0, :, :n, :n].cpu()   # (heads, n, n)
    fig, axes = plt.subplots(1, weights.shape[0], figsize=(5 * weights.shape[0], 5))
    for h, ax in enumerate(axes):
        sns.heatmap(weights[h], xticklabels=tokens, yticklabels=tokens, cmap="viridis", ax=ax, cbar=False)
        ax.set_title(f"Custom encoder – layer {layer + 1}, head {h + 1}")
        ax.tick_params(labelsize=7)
    plt.suptitle(f"True label: {LABELS[label]}")
    plt.tight_layout()
    plt.show()

def show_bert_attention(idx=0, layer=-1, heads=(0, 1, 2, 3)):
    ids, mask, label = val_data[idx]
    n = int(mask.sum())
    bert.eval()
    with torch.no_grad():
        out = bert(input_ids=ids[None, :n].to(device), attention_mask=mask[None, :n].to(device), output_attentions=True)
    tokens = tokenizer.convert_ids_to_tokens(ids[:n])
    att = out.attentions[layer][0].cpu()                                  # (heads, n, n)
    fig, axes = plt.subplots(1, len(heads), figsize=(5 * len(heads), 5))
    for ax, h in zip(axes, heads):
        sns.heatmap(att[h], xticklabels=tokens, yticklabels=tokens, cmap="viridis", ax=ax, cbar=False)
        ax.set_title(f"DistilBERT – last layer, head {h + 1}")
        ax.tick_params(labelsize=7)
    plt.suptitle(f"True label: {LABELS[label]}")
    plt.tight_layout()
    plt.show()

for i in [0, 1]:
    print(val_df.iloc[i]["premise"], " || ", val_df.iloc[i]["hypothesis"])
    show_custom_attention(i)
    show_bert_attention(i)

## 6. Reflection
*(Check the results table and attention maps from your run.)*

- **Accuracy:** NLI is a hard reasoning task: the model has to compare two sentences and decide whether one implies, contradicts, or is unrelated to the other. The **custom encoder**, trained from scratch on a few thousand English pairs, usually ends up only a little above the majority-class baseline (often **~40–50%**). It mostly picks up shallow cues such as word overlap or negation words ("not", "never"). **Fine-tuned DistilBERT** typically reaches **~65–75%**, because its pre-training on billions of words already taught it meaning, synonyms and sentence structure.
- **Attention maps:** the custom model's heads tend to be **diffuse or focused on frequent tokens** ([CLS], [SEP], punctuation): it hasn't had enough data to learn meaningful relationships. DistilBERT's heads show **clearer, specialised patterns**: some attend to the next or previous word, some to [SEP] (a known "no-op" pattern), and some link **matching or related words between premise and hypothesis**, which is effectively cross-sentence alignment inside self-attention.
- **Trade-offs:** the custom stack is tiny (a few million parameters, mostly embeddings), fast, fully transparent, and a great way to understand how attention works, but it needs huge amounts of data to be competitive. The pretrained model has ~66M parameters, needs a GPU and more time, and is harder to inspect, but it gives far better accuracy with little task data. In practice, **pre-training is what makes Transformers powerful**, not just the attention mechanism.
- **To improve the custom model:** more data (e.g. the full multilingual set or MNLI), initialising embeddings from a pre-trained model, more layers and heads, explicit **cross-attention** from hypothesis to premise, and longer training with learning-rate warm-up.